In [1]:
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values

DB_CONFIG = {
    "dbname": "sansilvestre_db",
    "user": "sansilvestre_user",
    "password":"secret",
    "host": "localhost",
    "port": 5433, #En el dockercompose es 5433 en vez de 5432 para no entrar en conflicto con un servicio postgres en local.
}

CSV_PATH = "./san_silvestre_silver.csv"  # ajustar ruta

In [2]:
df = pd.read_csv(CSV_PATH, sep=";")

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 40838 entries, 0 to 40837
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   edicion           40838 non-null  str    
 1   anho              40838 non-null  int64  
 2   carrera           40838 non-null  str    
 3   puesto            40838 non-null  int64  
 4   nombre            40838 non-null  str    
 5   dorsal            40838 non-null  int64  
 6   apellidos         40838 non-null  str    
 7   puesto_sexo       40761 non-null  float64
 8   puesto_categoria  40397 non-null  float64
 9   tiempo            40838 non-null  str    
 10  distancia_m       25899 non-null  float64
 11  ritmo_oficial     25899 non-null  str    
 12  ritmo_neto        23289 non-null  str    
 13  tiempo_neto       23289 non-null  str    
 14  categoria         40398 non-null  str    
 15  sexo              40776 non-null  str    
dtypes: float64(3), int64(3), str(10)
memory usage: 5.0 

In [4]:
SEXO_MAP = {
    "M": "masculino",
    "F": "femenino",
}
df["sexo"] = df["sexo"].map(SEXO_MAP)
df = df.astype(object).where(df.notna(), None)

In [5]:
conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()

In [6]:
df_ediciones = (
    df[["edicion", "anho"]]
    .drop_duplicates()
    .rename(columns={"edicion": "nombre", "anho": "ano"})
    .reset_index(drop=True)
)

In [7]:
edicion_map = {}

for _, row in df_ediciones.iterrows():
    cur.execute(
        """
        INSERT INTO edicion (nombre, ano)
        VALUES (%s, %s)
        RETURNING id
        """,
        (row["nombre"], int(row["ano"])),
    )
    edicion_id = cur.fetchone()[0]
    edicion_map[(row["nombre"], int(row["ano"]))] = edicion_id

conn.commit()

In [8]:
df_carreras = (
    df[["carrera", "distancia_m", "edicion", "anho"]]
    .drop_duplicates(subset=["carrera", "edicion", "anho"])
    .reset_index(drop=True)
)

In [9]:
carrera_map = {}

for _, row in df_carreras.iterrows():
    edicion_id = edicion_map[(row["edicion"], int(row["anho"]))]
    distancia = (int(row["distancia_m"]) if pd.notna(row["distancia_m"]) else None)
    
    cur.execute(
        """
        INSERT INTO carrera (nombre, distancia, edicion_id)
        VALUES (%s, %s, %s)
        RETURNING id
        """,
        (row["carrera"], distancia, edicion_id),
    )
    carrera_id = cur.fetchone()[0]
    carrera_map[(row["carrera"], row["edicion"], int(row["anho"]))] = carrera_id

conn.commit()

In [10]:
df_corredores = (
    df[["nombre", "apellidos", "sexo"]]
    .drop_duplicates(subset=["nombre", "apellidos"])
    .reset_index(drop=True))

In [11]:
corredor_map = {}

for _, row in df_corredores.iterrows():
    sexo = row["sexo"] if row["sexo"] in ("masculino", "femenino") else None

    cur.execute(
        """
        INSERT INTO corredor (nombre, apellidos, sexo)
        VALUES (%s, %s, %s)
        RETURNING id
        """,
        (row["nombre"], row["apellidos"], sexo),
    )
    corredor_id = cur.fetchone()[0]
    corredor_map[(row["nombre"], row["apellidos"])] = corredor_id

conn.commit()

In [17]:
INSERT_RESULTADO = """
    INSERT INTO resultado
        (categoria, dorsal, puesto, p_categoria, p_sexo,
         ritmo, ritmo_neto, tiempo, tiempo_neto, carrera_id)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    RETURNING id
"""

INSERT_CORREDOR_RESULTADO = """
    INSERT INTO corredor_resultado (corredor_id, resultado_id)
    VALUES (%s, %s)
"""

total = len(df)
BATCH_SIZE = 5000
count = 0

for i, (_, row) in enumerate(df.iterrows()):
    carrera_id = carrera_map[(row["carrera"], row["edicion"], int(row["anho"]))]
    corredor_id = corredor_map[(row["nombre"], row["apellidos"])]

    p_sexo = int(row["puesto_sexo"]) if row["puesto_sexo"] is not None else None
    p_cat = (int(row["puesto_categoria"]) if row["puesto_categoria"] is not None else None)
    dorsal = int(row["dorsal"]) if row["dorsal"] is not None else None
    puesto = int(row["puesto"]) if row["puesto"] is not None else None

    cur.execute(
        INSERT_RESULTADO,
        (
            row["categoria"],
            dorsal,
            puesto,
            p_cat,
            p_sexo,
            row["ritmo_oficial"],
            row["ritmo_neto"],
            row["tiempo"],
            row["tiempo_neto"],
            carrera_id,
        ),
    )
    resultado_id = cur.fetchone()[0]

    cur.execute(
        INSERT_CORREDOR_RESULTADO,
        (corredor_id, resultado_id),
    )
    count += 1
    if count % BATCH_SIZE == 0: conn.commit()
        
conn.commit()

In [18]:
cur.close()
conn.close()